# Optional Lab - Multi-class Classification (Fruit Sorting)


## 1.1 Goals
In this lab you will explore multi-class classification using neural networks, through a different, self-contained example: sorting fruit into categories based on two measured features.

By the end of this notebook you will be able to:
- Build a small multi-class neural network in TensorFlow/Keras
- Train it with `SparseCategoricalCrossentropy(from_logits=True)`
- Visualize the decision boundaries it learns
- Look inside the network, layer by layer, to see how it separates the classes

## 1.2 Tools
This notebook is fully self-contained — all plotting helper functions are defined directly in the cells below (no external `lab_utils` file is needed), so you can drop this notebook into any environment or GitHub repo and it will just run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_blobs
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

np.set_printoptions(precision=2)

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

%matplotlib inline

# 2.0 Multi-class Classification
Neural networks are often used to classify data into more than two categories. Examples include networks that:
- take in an image and classify it as {cat, dog, bird, other}
- take in a sentence and label each word as {noun, verb, adjective, ...}

A network built for this task has multiple units in its final layer — one per category. For a given input, the unit with the highest output value is the predicted class. If a softmax is applied to the outputs, they become probabilities of the input belonging to each class.

In this lab we'll build a small multi-class network in TensorFlow and then look inside it to see how it makes its predictions.

Let's start by creating a four-class **fruit sorting** data set.

## 2.1 Prepare and visualize our data
Imagine a simple fruit-sorting machine that measures two properties of each piece of fruit:
- **x0: Sweetness index** (higher = sweeter)
- **x1: Firmness index** (higher = firmer)

Four fruit types — **Grapes, Lemons, Apples, and Oranges** — cluster in different regions of this 2D feature space. We'll use `make_blobs` to synthesize a training set that mimics this scenario.

In [ ]:
# make a 4-class fruit-sorting dataset
class_names = ['Grape', 'Lemon', 'Apple', 'Orange']
classes = len(class_names)
m = 160
centers = [[2, 7], [7, 8], [8, 2], [2, 2]]   # sweetness, firmness cluster centers
std = 1.1

X_train, y_train = make_blobs(n_samples=m, centers=centers, cluster_std=std, random_state=42)

In [ ]:
def plt_fruit_data(X, y, class_names, ax=None, title="Fruit training data"):
    """Scatter plot of the training examples, colored by class."""
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(6, 5))
    cmap = plt.cm.get_cmap('tab10', len(class_names))
    for i, name in enumerate(class_names):
        idx = (y == i)
        ax.scatter(X[idx, 0], X[idx, 1], color=cmap(i), label=name,
                   edgecolor='k', s=45)
    ax.set_xlabel('x0 : Sweetness index')
    ax.set_ylabel('x1 : Firmness index')
    ax.set_title(title)
    ax.legend(loc='best')
    return ax

In [ ]:
plt_fruit_data(X_train, y_train, class_names)
plt.show()

Each dot is a training example. The axes (x0, x1) are the two input features, and the color shows which fruit class the example belongs to. Once trained, the model will be given a new (x0, x1) pair — a new piece of fruit — and will predict which class it belongs to.

This synthetic data set stands in for many real classification problems: several input features are used to predict one of several output categories.

In [ ]:
# show classes in data set
print(f"unique classes: {np.unique(y_train)}")
# show how classes are represented
print(f"class representation (first 10): {y_train[:10]}")
# show shapes of our dataset
print(f"shape of X_train: {X_train.shape}, shape of y_train: {y_train.shape}")

## 2.2 Model
This lab uses the same small 2-layer network structure as the original lab:
- **Layer 1**: 2 units, ReLU activation
- **Layer 2 (output)**: 4 units (one per fruit class), **linear** activation

Just like before, the output layer uses `linear` rather than `softmax`. It's more numerically stable to pass raw (logit) outputs into the loss function during training, and apply softmax afterward only if you need actual probabilities.

In [ ]:
tf.random.set_seed(2024)  # for reproducible results

model = Sequential(
    [
        Dense(2, activation='relu',   name="L1"),
        Dense(4, activation='linear', name="L2"),
    ]
)

The statements below compile and train the network. `from_logits=True` tells the loss function that the output layer produced raw linear values rather than softmax probabilities.

In [ ]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(0.01),
)

history = model.fit(
    X_train, y_train,
    epochs=200,
    verbose=0
)

print(f"final training loss: {history.history['loss'][-1]:.4f}")

With the model trained, let's see how it has classified the training data by plotting its decision boundaries.

In [ ]:
def plt_mc_decision_boundary(model, X, y, class_names, pad=1.5, n=300):
    """Plot the model's decision regions over the training data."""
    x0_min, x0_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    x1_min, x1_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    xx0, xx1 = np.meshgrid(np.linspace(x0_min, x0_max, n),
                            np.linspace(x1_min, x1_max, n))
    grid = np.c_[xx0.ravel(), xx1.ravel()]
    logits = model.predict(grid, verbose=0)
    preds = np.argmax(logits, axis=1).reshape(xx0.shape)

    cmap_bg = ListedColormap(plt.cm.tab10(np.arange(len(class_names))))
    fig, ax = plt.subplots(1, 1, figsize=(6.5, 5.5))
    ax.contourf(xx0, xx1, preds, alpha=0.25, cmap=cmap_bg,
                levels=np.arange(-0.5, len(class_names), 1))
    plt_fruit_data(X, y, class_names, ax=ax, title="Model decision boundaries")
    plt.show()

plt_mc_decision_boundary(model, X_train, y_train, class_names)

Above, the shaded regions show how the model has partitioned the 2D feature space into four decision regions. This simple model had little trouble separating the four fruit clusters. Let's look inside the network to understand how it did this.

In [ ]:
# gather the trained parameters from the first (hidden) layer
l1 = model.get_layer("L1")
W1, b1 = l1.get_weights()
print("W1:\n", W1)
print("b1:", b1)

In [ ]:
def plt_layer1_relu(X, y, W1, b1, class_names, pad=1.5, n=300):
    """Plot the ReLU output of each Layer-1 unit as a background heat map,
    with the training data overlaid."""
    x0_min, x0_max = X[:, 0].min() - pad, X[:, 0].max() + pad
    x1_min, x1_max = X[:, 1].min() - pad, X[:, 1].max() + pad
    xx0, xx1 = np.meshgrid(np.linspace(x0_min, x0_max, n),
                            np.linspace(x1_min, x1_max, n))
    grid = np.c_[xx0.ravel(), xx1.ravel()]

    n_units = W1.shape[1]
    fig, axes = plt.subplots(1, n_units, figsize=(6 * n_units, 5))
    if n_units == 1:
        axes = [axes]

    for j in range(n_units):
        z = grid @ W1[:, j] + b1[j]
        a = np.maximum(0, z).reshape(xx0.shape)
        im = axes[j].contourf(xx0, xx1, a, levels=20, cmap='viridis', alpha=0.85)
        axes[j].contour(xx0, xx1, a, levels=[0], colors='white', linewidths=2)
        plt_fruit_data(X, y, class_names, ax=axes[j], title=f"Layer 1, Unit {j} (ReLU output)")
        fig.colorbar(im, ax=axes[j])
    plt.tight_layout()
    plt.show()

plt_layer1_relu(X_train, y_train, W1, b1, class_names)

Each subplot shows the output of one Layer-1 unit across the whole feature space. Since these units use a ReLU, output is 0 on one side of a boundary line and grows on the other side (the white contour marks where the ReLU switches on). Together, the two units carve the space into quadrant-like regions that the output layer can then combine.

In [ ]:
# gather the trained parameters from the output layer
l2 = model.get_layer("L2")
W2, b2 = l2.get_weights()

# create the 'new features' -- the training examples after the Layer 1 transformation
Xl2 = np.maximum(0, np.dot(X_train, W1) + b1)
print("Xl2 shape:", Xl2.shape)

In [ ]:
def plt_output_layer_linear(Xl2, y, W2, b2, class_names, pad=1.0, n=300):
    """Plot the linear output of each output-layer unit over the Layer-1-transformed
    feature space (Xl2), with the transformed training points overlaid."""
    x0_min, x0_max = -pad, Xl2[:, 0].max() + pad
    x1_min, x1_max = -pad, Xl2[:, 1].max() + pad
    xx0, xx1 = np.meshgrid(np.linspace(x0_min, x0_max, n),
                            np.linspace(x1_min, x1_max, n))
    grid = np.c_[xx0.ravel(), xx1.ravel()]

    n_units = W2.shape[1]
    fig, axes = plt.subplots(1, n_units, figsize=(4.5 * n_units, 4.5))
    cmap_pts = plt.cm.get_cmap('tab10', len(class_names))

    for j in range(n_units):
        z = grid @ W2[:, j] + b2[j]
        z = z.reshape(xx0.shape)
        im = axes[j].contourf(xx0, xx1, z, levels=20, cmap='plasma', alpha=0.85)
        for i, name in enumerate(class_names):
            idx = (y == i)
            axes[j].scatter(Xl2[idx, 0], Xl2[idx, 1], color=cmap_pts(i),
                            edgecolor='k', s=45, label=name if j == 0 else None)
        axes[j].set_title(f"Output Unit {j}: {class_names[j]} score")
        axes[j].set_xlabel(r'$a^{[1]}_0$')
        axes[j].set_ylabel(r'$a^{[1]}_1$')
        fig.colorbar(im, ax=axes[j])
    fig.legend(loc='lower center', ncol=len(class_names), bbox_to_anchor=(0.5, -0.05))
    plt.tight_layout()
    plt.show()

plt_output_layer_linear(Xl2, y_train, W2, b2, class_names)

## Explanation
#### Layer 1
These plots show what each Layer-1 unit computes across the (sweetness, firmness) space. Because they use a ReLU, each unit's output is exactly zero on one side of a line and grows past it on the other side (the white contour marks that boundary). In effect, each unit acts like a simple linear "cut" through the data, separating two groups of fruit classes from the other two.

#### Layer 2, the output layer
The dots in these plots are the training examples **after** being passed through Layer 1 — this is the new 2D feature space ($a^{[1]}_0, a^{[1]}_1$) that Layer 1 has built. Each output unit then computes a simple linear score over this new space, and its background color shows where that score is highest. During training, the four fruit classes ended up occupying different corners of this transformed space, and each output unit specialized in recognizing one corner.

An important detail that isn't obvious from a single unit's plot: it's not enough for a unit to produce a high value for *its* class — it must also be **higher than the other three units' outputs** at that point. That competition between units is enforced by the implied softmax inside `SparseCategoricalCrossentropy`, which compares all four outputs together during training.

You don't need to understand every internal detail to use neural networks effectively — but hopefully seeing it laid out like this gives you a bit more intuition for what's happening under the hood.

## Try it yourself
A few ideas to extend this notebook:
- Change `centers` and `std` in section 2.1 to make the classes overlap more — watch how the decision boundaries and training loss change.
- Add a 5th fruit class and a 5th output unit.
- Increase the number of Layer 1 units (e.g. to 4 or 8) and see how the decision boundaries become more flexible.
- Split the data into train/test sets and check the model's accuracy on unseen fruit.

## Congratulations!
You built and trained a multi-class neural network in TensorFlow, visualized its decision boundaries, and looked inside it layer by layer to see how it separates classes.